# Financial Market ETL Pipeline

## Project Overview
This notebook demonstrates an end-to-end ETL pipeline for financial data.
The pipeline extracts data from multiple APIs, processes it using Python, 
stores it in PostgreSQL and visualizes it in Power BI.

## Install Environment Setup and Libraries

In [1]:
!pip install psycopg2-binary sqlalchemy

## Data Extraction from Multiple APIs

In [2]:
import requests

API_KEY = "UWS0Q8T4EYLGA0MX"
symbol = "ALV.DE"

url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={API_KEY}"

response = requests.get(url)
stock_data = response.json()

print(stock_data.keys())

dict_keys(['Meta Data', 'Time Series (Daily)'])


In [3]:
import requests
import pandas as pd

# Get EUR exchange rates
url = "https://api.frankfurter.app/latest?from=EUR"

response = requests.get(url)
fx_data = response.json()

print(fx_data)

{'amount': 1.0, 'base': 'EUR', 'date': '2026-06-19', 'rates': {'AUD': 1.6348, 'BRL': 5.9173, 'CAD': 1.6228, 'CHF': 0.9248, 'CNY': 7.7624, 'CZK': 24.227, 'DKK': 7.4746, 'GBP': 0.86653, 'HKD': 8.9887, 'HUF': 352.68, 'IDR': 20420.55, 'ILS': 3.3968, 'INR': 108.1675, 'ISK': 144.0, 'JPY': 184.88, 'KRW': 1757.1, 'MXN': 19.8796, 'MYR': 4.7439, 'NOK': 11.1045, 'NZD': 1.9967, 'PHP': 69.645, 'PLN': 4.2615, 'RON': 5.2396, 'SEK': 10.974, 'SGD': 1.4804, 'THB': 37.681, 'TRY': 53.2587, 'USD': 1.1467, 'ZAR': 18.8907}}


In [4]:
import requests

# Germany GDP data
url = "https://api.worldbank.org/v2/country/DEU/indicator/NY.GDP.MKTP.CD?format=json"

response = requests.get(url)
macro_data = response.json()

# Check data
print(macro_data[:2])

[{'page': 1, 'pages': 2, 'per_page': 50, 'total': 66, 'sourceid': '2', 'lastupdated': '2026-04-08'}, [{'indicator': {'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (current US$)'}, 'country': {'id': 'DE', 'value': 'Germany'}, 'countryiso3code': 'DEU', 'date': '2025', 'value': None, 'unit': '', 'obs_status': '', 'decimal': 0}, {'indicator': {'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (current US$)'}, 'country': {'id': 'DE', 'value': 'Germany'}, 'countryiso3code': 'DEU', 'date': '2024', 'value': 4685592577804.69, 'unit': '', 'obs_status': '', 'decimal': 0}, {'indicator': {'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (current US$)'}, 'country': {'id': 'DE', 'value': 'Germany'}, 'countryiso3code': 'DEU', 'date': '2023', 'value': 4562207532490.28, 'unit': '', 'obs_status': '', 'decimal': 0}, {'indicator': {'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (current US$)'}, 'country': {'id': 'DE', 'value': 'Germany'}, 'countryiso3code': 'DEU', 'date': '2022', 'value': 4201021706478.62, 'unit': '', 'obs_status': '', 'decimal'

## Data Transformation

In [5]:
import pandas as pd

time_series = stock_data['Time Series (Daily)']

df = pd.DataFrame.from_dict(time_series, orient='index')
df.columns = ['open', 'high', 'low', 'close', 'volume']

df.index = pd.to_datetime(df.index)
df = df.astype(float)
df = df.sort_index()

df.head()

,open,high,low,close,volume
2026-01-28,365.1,368.3,364.5,367.9,390928.0
2026-01-29,369.3,372.9,367.6,369.3,479417.0
2026-01-30,371.1,372.6,370.1,371.8,587457.0
2026-02-02,373.0,379.8,372.5,379.2,491075.0
2026-02-03,380.0,382.7,378.7,382.4,456406.0


## Data Cleaning

In [6]:
df.dropna(inplace=True)

df['daily_return'] = df['close'].pct_change()
df['moving_avg_7'] = df['close'].rolling(7).mean()
df['volatility'] = df['daily_return'].rolling(7).std()

df.head()

,open,high,low,close,volume,daily_return,moving_avg_7,volatility
2026-01-28,365.1,368.3,364.5,367.9,390928.0,NaN,NaN,NaN
2026-01-29,369.3,372.9,367.6,369.3,479417.0,0.003805,NaN,NaN
2026-01-30,371.1,372.6,370.1,371.8,587457.0,0.006770,NaN,NaN
2026-02-02,373.0,379.8,372.5,379.2,491075.0,0.019903,NaN,NaN
2026-02-03,380.0,382.7,378.7,382.4,456406.0,0.008439,NaN,NaN


In [7]:
# Convert to DataFrame
fx_df = pd.DataFrame(fx_data['rates'], index=[0])

# Add date column
fx_df['date'] = fx_data['date']

# Reorder
fx_df = fx_df.set_index('date')

fx_df.head()

,AUD,BRL,CAD,CHF,CNY,CZK,DKK,GBP,HKD,HUF,...,NZD,PHP,PLN,RON,SEK,SGD,THB,TRY,USD,ZAR
date,,,,,,,,,,,,,,,,,,,,,
2026-06-19,1.6348,5.9173,1.6228,0.9248,7.7624,24.227,7.4746,0.86653,8.9887,352.68,...,1.9967,69.645,4.2615,5.2396,10.974,1.4804,37.681,53.2587,1.1467,18.8907


In [8]:
import pandas as pd

# Extract actual dataset
data = macro_data[1]

# Convert to DataFrame
macro_df = pd.DataFrame(data)

# Keep important columns
macro_df = macro_df[['date', 'value']]

# Rename columns
macro_df.columns = ['year', 'gdp']

# Remove missing values
macro_df.dropna(inplace=True)

# Convert year to integer
macro_df['year'] = macro_df['year'].astype(int)

# Sort by year
macro_df = macro_df.sort_values(by='year')

macro_df.head()

,year,gdp
49,1976,5.216587e+11
48,1977,6.026983e+11
47,1978,7.431829e+11
46,1979,8.845742e+11
45,1980,9.537725e+11


## Data Storage in PostgreSQL

In [9]:
from sqlalchemy import create_engine

username = "postgres"
password = "5678"
host = "localhost"
port = "5432"
database = "financial_db"

# Create connection
engine = create_engine(f'postgresql://{username}:{password}@{host}:{port}/{database}')

print("✅ Connected to PostgreSQL")


✅ Connected to PostgreSQL


In [10]:
df.to_sql('market_prices', engine, if_exists='replace')

print("✅ Data successfully loaded into PostgreSQL")


✅ Data successfully loaded into PostgreSQL


In [11]:
fx_df.to_sql('fx_rates', engine, if_exists='replace')

print("✅ FX data loaded into PostgreSQL")

✅ FX data loaded into PostgreSQL


In [12]:
macro_df.to_sql('macro_indicators', engine, if_exists='replace')

print("✅ Macro data loaded into PostgreSQL")

✅ Macro data loaded into PostgreSQL


## Machine Learning Component

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Remove missing values
df_ml = df.dropna()

# Select numeric data
df_numeric = df_ml.select_dtypes(include=['number'])

# Define X and y
X = df_numeric.iloc[:, :-1]
y = df_numeric.iloc[:, -1]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train
model = RandomForestRegressor()
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)

print("Mean Squared Error:", mse)

Mean Squared Error: 2.0317644155504296e-05
